In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType
from itertools import chain
from pyspark.sql import Window

# Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.transtractions")

# Data Transform

##Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

##Normalized the date

In [0]:
(df.select(F.regexp_replace(F.regexp_replace(F.trim("order_date"), "[0-9]", "d"), "[A-Za-z]", "a").alias("pattern"))
   .groupBy("pattern").count().orderBy(F.desc("count")).show(truncate=False))

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

In [0]:
df = df.withColumn(
    "order_date",
    F.coalesce(
        F.try_to_date("order_date", "yyyy-MM-dd"),
        F.try_to_date("order_date", "yyyy-dd-MM"),
        F.try_to_date("order_date", "dd/MM/yyyy"),
        F.try_to_date("order_date", "MM/dd/yyyy")
    ))

In [0]:
(df.select(F.regexp_replace(F.regexp_replace(F.trim("order_date"), "[0-9]", "d"), "[A-Za-z]", "a").alias("pattern"))
   .groupBy("pattern").count().orderBy(F.desc("count")).show(truncate=False))

##Standerdize string columns

### Format the store name

In [0]:
df = df.withColumn("store_name", F.initcap("store_name"))

###Normalized the store name

In [0]:
df.select("store_id","store_name").distinct().orderBy("store_id").show(100, truncate=False )

In [0]:
mapping ={
    "Dwntwn Flagship": "Downtown Flagship",
    "Lakeview Ctr": "Lakeview Center",
    "Magnolia Cmns": "Magnolia Commons",
    "Peachtree Cnrs": "Peachtree Corners"
}

df = df.replace(mapping, subset=["store_name"])


In [0]:
df.select("store_id","store_name").distinct().orderBy("store_id").show(100, truncate=False )

In [0]:
cols = ["store_id", "store_name", "region", "product_name", "category", "loyalty_member"]
for c in cols:
    print(f"----{c}-----")
    df.select(c).distinct().show(50, truncate=False)

###Normalized the product name

In [0]:
df.select("product_id","product_name").distinct().orderBy("product_name").show(100, truncate=False )

In [0]:
df = df.withColumn(
    "product_name",
    F.when(F.col("product_id") == "P010", "Rain Jacket - Standerd")
     .when(F.col("product_id") == "P001", "Rain Jacket - Primium")
     .otherwise(F.col("product_name").cast("string"))
)

In [0]:
df.select("product_id","product_name", "product_name").distinct().orderBy("product_id").show(100, truncate=False )

###Standerdize the Unit Price

In [0]:
(df.groupBy("product_id", "product_name", "unit_price")
   .count()
   .orderBy("product_name", "unit_price")
   .show(100, truncate=False))

In [0]:
w = Window.partitionBy("product_id")

df = (df
    .withColumn("median_price", F.expr("percentile_approx(unit_price, 0.5)").over(w))
    .withColumn("ratio", F.col("unit_price") / F.col("median_price"))
    .withColumn(
        "is_outlier",
        F.coalesce(
            (F.col("ratio") > 3) | (F.col("ratio") < 1/3),
            F.lit(False)
        )
    )
)

# df = df.drop("ratio", "median_price")

In [0]:
df.filter(F.col("is_outlier")=='true').show()

In [0]:
df = (df
    .withColumn(
        "unit_price",
        F.when(F.col("is_outlier"), F.col("median_price")).otherwise(F.col("unit_price"))
    )
    .drop("median_price", "ratio","is_outlier")
)

In [0]:
df.filter(F.col("is_outlier")=='true').show()

In [0]:
df.filter(F.col("is_outlier") == True).show()

###Standardized loyalty column

In [0]:
df = df.withColumn(
    "loyalty_member",
    F.when(F.col("loyalty_member") == 1, "yes")
     .when(F.col("loyalty_member") == 0, "no")
     .otherwise(F.col("loyalty_member").cast("string"))
)

##Manage Duplicates

In [0]:
dups = (df.groupBy(df.columns)
          .count()
          .filter("count > 1")
          .orderBy(F.desc("count")))

dups.show(1000, truncate=False)
print("duplicated groups:", dups.count())

In [0]:
w = Window.partitionBy(*df.columns).orderBy(F.lit(1))
df = df.withColumn("row_count", F.row_number().over(w))
df = df.filter("row_count = 1").drop("row_count")   

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

#Write Into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.transtractions")
)

In [0]:
%sql
SELECT * FROM silver.transtractions